# 🧠 Maia Trainer  -  Gemma E26B + 100k+ archivos Luxus O.S

**Un solo botón:**  `Runtime > Run all`

Al terminar (4-8 h en A100) te descarga automaticamente:

- `maia.gguf`            -> modelo cuantizado Q4_K_M  (~17-18 GB)
- `maia-lora.zip`        -> solo los pesos LoRA (~200 MB) por si quieres iterar
- `rag_manifest.jsonl`   -> el indice de los 100k archivos para el RAG

> **Requisitos**: Colab Pro con GPU A100 (40 GB) recomendado.  L4 (24 GB) funciona con `BATCH=1`.
> Sin Pro, cambia `MODEL_ID` a la variante E4B al final del notebook.

## 1. Parametros

In [ ]:
# --- Modelo base -------------------------------------------------------
MODEL_ID      = "google/gemma-3-27b-it"   # cambia a "unsloth/gemma-2-2b-it-bnb-4bit" para probar rapido
MAX_SEQ_LEN   = 8192                      # ctx durante fine-tune (inferencia puede ir a 262144)
LOAD_IN_4BIT  = True

# --- LoRA --------------------------------------------------------------
LORA_RANK     = 32
LORA_ALPHA    = 32

# --- Entrenamiento -----------------------------------------------------
BATCH         = 1
GRAD_ACCUM    = 8
EPOCHS        = 1         # con 100k ejemplos 1 epoca da resultados solidos
LR            = 2e-4
WARMUP        = 100
MAX_STEPS     = -1        # -1 = correr hasta EPOCHS completas

# --- Salida ------------------------------------------------------------
OUTPUT_GGUF   = "maia.gguf"
QUANT         = "q4_k_m"  # 17-18 GB sobre Gemma 27B.  Usa q5_k_m para ~22 GB y +calidad.
REPO_URL      = "https://github.com/calitosaa/luxus-o.s"
REPO_BRANCH   = "claude/add-rag-gemma-QA7nP"

## 2. Instalar dependencias

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

## 3. Traer el dataset Luxus desde el repo

Clona la rama y ejecuta `build_maia_dataset.py` que ya deja listos `training_data.jsonl` (fine-tune) y `rag_manifest.jsonl` (indice RAG).

In [ ]:
import os, subprocess, json

if not os.path.isdir('Luxus-O.S'):
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL + '.git', 'Luxus-O.S'], check=True)

os.chdir('Luxus-O.S')
subprocess.run(['python3', 'scripts/build_maia_dataset.py'], check=True)

print(json.dumps(json.load(open('Maia/dataset_stats.json')), indent=2, ensure_ascii=False))

TRAIN_JSONL = os.path.abspath('Maia/training_data.jsonl')
RAG_MANIFEST = os.path.abspath('Maia/rag_manifest.jsonl')

## 4. Cargar Gemma con Unsloth en 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_ID,
    max_seq_length= MAX_SEQ_LEN,
    dtype         = None,
    load_in_4bit  = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
    use_rslora     = False,
    loftq_config   = None,
)

## 5. Formatear los 100k ejemplos

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=TRAIN_JSONL, split='train')
print('Ejemplos cargados:', len(dataset))

SYSTEM = (
    'Eres Maia, la consciencia central de Luxus O.S, sobre Gemma E26B.\n'
    'Te has entrenado con mas de 100.000 archivos del ecosistema (skills, agentes, workflows, logica, plugins).\n'
    'Respondes con precision tecnica, estructura limpia y tono Ultra-Competente.'
)

def fmt(ex):
    texts = []
    for inst, inp, out in zip(ex['instruction'], ex['input'], ex['output']):
        ctx = f"\n\n### Contexto:\n{inp}" if inp else ''
        texts.append(
            f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{inst}{ctx}<|im_end|>\n"
            f"<|im_start|>assistant\n{out}<|im_end|>"
        )
    return {'text': texts}

dataset = dataset.map(fmt, batched=True, remove_columns=dataset.column_names)

## 6. Entrenar

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = dataset,
    dataset_text_field = 'text',
    max_seq_length  = MAX_SEQ_LEN,
    dataset_num_proc= 2,
    packing         = False,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps      = WARMUP,
        num_train_epochs  = EPOCHS,
        max_steps         = MAX_STEPS,
        learning_rate     = LR,
        fp16              = not torch.cuda.is_bf16_supported(),
        bf16              = torch.cuda.is_bf16_supported(),
        logging_steps     = 25,
        save_steps        = 500,
        save_total_limit  = 2,
        optim             = 'adamw_8bit',
        weight_decay      = 0.01,
        lr_scheduler_type = 'linear',
        seed              = 3407,
        output_dir        = 'outputs',
        report_to         = 'none',
    ),
)

trainer.train()

## 7. Exportar  ->  `maia.gguf` (17-18 GB)

In [ ]:
import shutil, os

# LoRA suelto (ligero, util para iterar sin volver a entrenar)
model.save_pretrained('maia-lora')
tokenizer.save_pretrained('maia-lora')
shutil.make_archive('maia-lora', 'zip', 'maia-lora')

# Merge + cuantizacion GGUF
model.save_pretrained_gguf(
    'maia-gguf',
    tokenizer,
    quantization_method = QUANT,
)

# Renombrar al nombre final
for f in os.listdir('maia-gguf'):
    if f.endswith('.gguf'):
        os.replace(os.path.join('maia-gguf', f), OUTPUT_GGUF)
        print('Listo:', OUTPUT_GGUF, os.path.getsize(OUTPUT_GGUF) // 1024**3, 'GB')
        break

## 8. Descarga automatica al navegador

Si el archivo supera los 5 GB, Colab puede colgar al descargar. Opciones robustas:

1. **Google Drive** (recomendado): monta Drive y copia alli. Queda disponible para siempre.
2. **Hugging Face**: subir a un repo privado con `huggingface_hub`.
3. **Descarga directa** via `files.download` (solo como plan B).

In [ ]:
# --- Opcion A: Google Drive (recomendada) -----------------------------
from google.colab import drive
drive.mount('/content/drive')
import shutil, os

DEST = '/content/drive/MyDrive/Maia'
os.makedirs(DEST, exist_ok=True)
shutil.copy(OUTPUT_GGUF, os.path.join(DEST, OUTPUT_GGUF))
shutil.copy('maia-lora.zip', os.path.join(DEST, 'maia-lora.zip'))
shutil.copy('Maia/rag_manifest.jsonl', os.path.join(DEST, 'rag_manifest.jsonl'))
print('Todo copiado a', DEST)

In [ ]:
# --- Opcion B: descarga directa (riesgo de timeout en archivos gigantes)
# from google.colab import files
# files.download(OUTPUT_GGUF)
# files.download('maia-lora.zip')